# BSM CORRECTED RUN — Notebook 06: ResNet-18 (split-by-original, train-aug only)

**NON-PAPER-FAITHFUL.** Group-split by original image; augment only train originals; train = originals+aug; eval on originals.

In [1]:
import os, sys, random, json, pathlib, time
from pathlib import Path
import numpy as np
import torch
from PIL import Image
import torchvision.transforms.functional as TF
from sklearn.model_selection import GroupShuffleSplit
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"): drive.mount("/content/drive")
PROJECT = "/content/drive/MyDrive/bsm_paper_replication"
REPO = Path("/content/BSM_Project_2")
if str(REPO) not in sys.path: sys.path.insert(0, str(REPO))
if PROJECT not in sys.path: sys.path.insert(0, PROJECT)
for sub in ["data","augmented_corr","checkpoints_corr","results_corr"]:
    Path(PROJECT)/sub
CLASSES = ["Ilmenite","Rutile","Zircon","Garnet","Monazite","Sillimanite"]
CLASS_TO_IDX = {c:i for i,c in enumerate(CLASSES)}
print("gpu:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

gpu: True Tesla T4


In [2]:
ORIG = Path(REPO)/"data"/"originals"
orig_items = []
for c in CLASSES:
    d = ORIG/c
    for p in sorted(d.glob("*")):
        if p.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"]:
            orig_items.append({"path": str(p), "cls": c, "y": CLASS_TO_IDX[c], "gid": p.stem})
paths = [x["path"] for x in orig_items]; ys = np.array([x["y"] for x in orig_items]); gids = np.array([x["gid"] for x in orig_items])
gss1 = GroupShuffleSplit(n_splits=1, train_size=0.7, random_state=SEED)
tr_idx, rest_idx = next(gss1.split(paths, ys, groups=gids))
gids_rest = gids[rest_idx]; ys_rest = ys[rest_idx]
gss2 = GroupShuffleSplit(n_splits=1, train_size=0.5, random_state=SEED)
va_rel, te_rel = next(gss2.split(np.array(paths)[rest_idx], ys_rest, groups=gids_rest))
va_idx = rest_idx[va_rel]; te_idx = rest_idx[te_rel]
train_orig = [orig_items[i] for i in tr_idx]; val_orig = [orig_items[i] for i in va_idx]; test_orig = [orig_items[i] for i in te_idx]
def pil_to_tensor(im): im = TF.resize(im,(224,224)); return TF.to_tensor(im)
def aug_variants(img_pil, n=22):
    out=[]; img=TF.resize(img_pil,(256,256))
    for _ in range(n):
        im=img
        if random.random()<0.5: im=TF.hflip(im)
        if random.random()<0.5: im=TF.vflip(im)
        im=TF.rotate(im, random.uniform(-15,15))
        im=TF.center_crop(TF.resize(im,(256,256)),224)
        if random.random()<0.3: im=TF.adjust_brightness(im,random.uniform(0.8,1.2))
        if random.random()<0.3: im=TF.adjust_contrast(im,random.uniform(0.8,1.2))
        if random.random()<0.3: im=TF.adjust_saturation(im,random.uniform(0.8,1.2))
        out.append(im)
    return out
random.seed(SEED); aug_list=[]
for it in train_orig:
    pil=Image.open(it['path']).convert('RGB')
    for im in aug_variants(pil,22): aug_list.append({'pil':im,'y':it['y']})
Xtr_list,ytr_list=[],[]
for it in train_orig:
    pil=Image.open(it['path']).convert('RGB'); Xtr_list.append(pil_to_tensor(pil)); ytr_list.append(it['y'])
for it in aug_list: Xtr_list.append(pil_to_tensor(it['pil'])); ytr_list.append(it['y'])
Xtr=torch.stack(Xtr_list); ytr=torch.tensor(ytr_list,dtype=torch.long)
Xva_list,yva_list=[],[]
for it in val_orig:
    pil=Image.open(it['path']).convert('RGB'); Xva_list.append(pil_to_tensor(pil)); yva_list.append(it['y'])
Xva=torch.stack(Xva_list); yva=torch.tensor(yva_list,dtype=torch.long)
Xte_list,yte_list=[],[]
for it in test_orig:
    pil=Image.open(it['path']).convert('RGB'); Xte_list.append(pil_to_tensor(pil)); yte_list.append(it['y'])
Xte=torch.stack(Xte_list); yte=torch.tensor(yte_list,dtype=torch.long)
print('train:',Xtr.shape[0],'val:',Xva.shape[0],'test:',Xte.shape[0])

In [3]:
from models import resnet18
import common
MICRO=32
train_loader=DataLoader(TensorDataset(Xtr,ytr),batch_size=MICRO,shuffle=False)
val_loader=DataLoader(TensorDataset(Xva,yva),batch_size=MICRO,shuffle=False)
torch.manual_seed(SEED)
model=resnet18().cuda()
print('corr resnet18 params:', f'{sum(p.numel() for p in model.parameters()):,}')
CKPT=Path(PROJECT)/'checkpoints_corr'/'resnet18'; RES=Path(PROJECT)/'results_corr'
CKPT.mkdir(parents=True,exist_ok=True); RES.mkdir(parents=True,exist_ok=True)
t0=time.time()
hist=common.train_model(model,train_loader,val_loader,torch.device('cuda'),epochs=60,full_batch=True,ckpt_dir=CKPT,history_path=RES/'history_resnet18_corrected.json',tag='resnet18_corr',lr=1e-3,resume=True)
print(f'TRAINING DONE in {(time.time()-t0)/60:.1f} min')
print('best epoch:',hist.get('best_epoch'),'best val:',hist.get('best_val_acc'))

corr resnet18 params: 11,689,512


In [4]:
device=torch.device('cuda')
model=resnet18().to(device)
ck=torch.load(CKPT/'resnet18_corr_best.pt',map_location=device,weights_only=True)
model.load_state_dict(ck)
_hist=json.load(open(RES/'history_resnet18_corrected.json'))
model.eval()
test_loader=DataLoader(TensorDataset(Xte,yte),batch_size=32,shuffle=False)
all_preds,all_true=[],[]
with torch.no_grad():
    for xb,yb in test_loader:
        logits=model(xb.to(device))
        all_preds.append(logits.argmax(1).cpu().numpy()); all_true.append(yb.numpy())
y_pred=np.concatenate(all_preds); y_true=np.concatenate(all_true)
acc=accuracy_score(y_true,y_pred)
prec,rec,f1,_=precision_recall_fscore_support(y_true,y_pred,average='macro',zero_division=0)
cm=confusion_matrix(y_true,y_pred,labels=range(6))
spec_list=[]
for i in range(6):
    tn=cm.sum()-cm[i,:].sum()-cm[:,i].sum()+cm[i,i]
    fp=cm[:,i].sum()-cm[i,i]
    spec_list.append(tn/(tn+fp) if (tn+fp)>0 else 0.0)
spec=float(np.mean(spec_list))
print(f'accuracy {acc:.4f} | macro F1 {f1:.4f} | spec {spec:.4f}')
results={"model":"resnet18","type":"CORRECTED","notebook":"06_corr_resnet18","best_epoch":int(_hist.get('best_epoch',-1)),"test_split":{"accuracy":float(acc),"macro_precision":float(prec),"macro_recall":float(rec),"macro_f1":float(f1),"macro_specificity":spec,"confusion_matrix":cm.tolist(),"n_test":int(len(y_true))}}
with open(RES/'metrics_resnet18_corrected.json','w') as f: json.dump(results,f,indent=2)
print('saved')

accuracy: 0.9091


In [5]:
import glob, os, shutil
os.makedirs(os.path.join(PROJECT,'notebooks'),exist_ok=True)
cands=glob.glob('/content/drive/MyDrive/Colab Notebooks/*.ipynb')
if cands:
    newest=max(cands,key=os.path.getmtime)
    shutil.copy(newest,os.path.join(PROJECT,'notebooks','06_corr_resnet18.ipynb'))
    print('copied')

copied
